# Scope Drift Analysis

Run the scope drift model on citation networks. Configure the parameters below before running.

In [1]:
# ============================================================
# CONFIGURATION - Modify these settings before running
# ============================================================
import os

# Year range for network construction
os.environ["START_YEAR"] = "2025"
os.environ["END_YEAR"] = "2025"

# Network mode: "ego", "full", or "global"
# - "ego": Frontiers papers + direct citations
# - "full": Frontiers + related journals (default)
# - "global": ALL publications from ALL publishers (requires 64GB+ RAM)
os.environ["NETWORK_MODE"] = "global"

# Journals to analyze
# Option 1: Top N Frontiers journals by publication count
os.environ["TOP_N_JOURNALS"] = "5"

# Option 2: Specific journal IDs (uncomment to use)
# os.environ["JOURNAL_IDS"] = "910533066753,2972117368834,2379411881984,3315714752512,2405181685761"

# Clustering level for OOS detection: "macro", "meso", or "micro"
os.environ["JOURNAL_DRIFT_LEVEL"] = "meso"

# Leiden resolution - higher = more, smaller clusters (default meso: 0.000006)
os.environ["LEIDEN_RESOLUTION_MESO"] = "0.001"
os.environ["MIN_COMMUNITY_SIZE"] = "50"

# ---- Edge Weight Tuning (for better weight distribution) ----
# Lower tau = older citations weigh much less (increases variance)
os.environ["TEMPORAL_DECAY_TAU"] = "3.0"  # default 5.0, try 2.0-3.0

# Lower = more penalty for same-journal citations (breaks up journal cliques)
os.environ["SELF_CITE_JOURNAL_WEIGHT"] = "0.2"  # default 0.5, try 0.2-0.3

# Higher = only strong bibliographic coupling edges (clearer signal)
os.environ["BC_MIN_SHARED_REFS"] = "2"  # default 3, try 5-8

# Weight transform to spread distribution: "none", "log", "sqrt", or "power:0.3"
os.environ["WEIGHT_TRANSFORM"] = "log"  # log works well for spreading weights

# Remove weak edges below this threshold (creates clearer community boundaries)
os.environ["EDGE_WEIGHT_THRESHOLD"] = "0.05"  # try 0.05-0.2

# Contrast: amplify differences (>1 = strong edges stronger, weak edges weaker)
os.environ["WEIGHT_CONTRAST"] = "1.0"  # try 1.5-3.0

print("Configuration set:")
print(f"  Year range: {os.environ['START_YEAR']} - {os.environ['END_YEAR']}")
print(f"  Network mode: {os.environ['NETWORK_MODE']}")
print(f"  Top N journals: {os.environ['TOP_N_JOURNALS']}")

Configuration set:
  Year range: 2025 - 2025
  Network mode: global
  Top N journals: 5


In [2]:
# ============================================================
# RUN SCOPE DRIFT ANALYSIS
# ============================================================
import importlib.util
from pathlib import Path

# Get the directory where this notebook is located (works on Windows & Linux)
notebook_dir = Path().absolute()
module_path = notebook_dir / "src" / "scope_drift.py"

print(f"Loading module from: {module_path}")

# Load scope_drift module
spec = importlib.util.spec_from_file_location("scope_drift", module_path)
scope_drift = importlib.util.module_from_spec(spec)
spec.loader.exec_module(scope_drift)

# Run the analysis
results = scope_drift.main()

2026-06-09 09:50:39,929 [INFO] ============================================================
2026-06-09 09:50:39,930 [INFO] SCOPE DRIFT — Global Citation Network Analysis
2026-06-09 09:50:39,930 [INFO]   Mode: GLOBAL, Years: 2025-2025
2026-06-09 09:50:39,931 [INFO] ============================================================
2026-06-09 09:50:39,931 [INFO] Getting top 5 Frontiers journals by publication count...


Loading module from: /home/jupyter/scope-drift-model/src/scope_drift.py
Loading module from: /home/jupyter/scope-drift-model/src/create_html_output.py


/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
2026-06-09 09:50:41,325 [INFO] Top Frontiers journals:
    JournalId                DisplayName  pubs
3315714752512    Frontiers in Immunology  7031
2379411881984 Frontiers in Public Health  5426
2826088480769      Frontiers in Medicine  4837
2972117368834      Frontiers in Oncology  4576
2405181685761    Frontiers in Psychology  4267
2026-06-09 09:50:41,325 [INFO] Fetching Frontiers publication IDs...
/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
2026-06-09 09:50:42,887 [INFO] Frontiers publications: 26,137
2026-06-09 09:50:42,888 [INFO] ============================================================
2026-06-09 09:50:42,888 [INF

BadRequest: 400 POST https://bigquery.googleapis.com/bigquery/v2/projects/ocean-tech-adv-analytics-p-usr/jobs?prettyPrint=false: The query is too large (3133.49K characters, 2160118 characters over the limit). The maximum standard SQL query length is 1024.00K characters, including comments and white space characters.

Location: None
Job ID: a5521eea-4a1c-4f66-8f30-e7b089b04fc7


In [ ]:
import json
from pathlib import Path
output_dir = Path("output")
# Load the JSON results
json_file = output_dir / "scope_global_network.json"
if json_file.exists():
    with open(json_file) as f:
        saved_results = json.load(f)
    print(f"Loaded results from {json_file}")
    print(f"  Journals analyzed: {len(saved_results['journals'])}")
    print(f"  Communities: {len(saved_results['communities'])}")
    
    # Show top communities by size
    print("\n" + "="*60)
    print("TOP 20 COMMUNITIES BY SIZE")
    print("="*60)
    communities = saved_results['communities']
    # Sort by size (they should already be sorted, but just in case)
    communities_sorted = sorted(communities, key=lambda x: -x['size'])
    
    for i, comm in enumerate(communities_sorted[:20]):
        print(f"{i+1:2}. {comm['label'][:50]:<50} | Size: {comm['size']:>8,} | Frontiers: {comm['frontiers_pct']:>5.1f}%")
else:
    print(f"No results file found at {json_file}")
    print("Run the analysis first.")

In [ ]:
# ============================================================
# LOAD OUTPUT FILES (if viewing results later)
# ============================================================
import json
from pathlib import Path

output_dir = Path("output")

# Load the JSON results
json_file = output_dir / "scope_global_network.json"
if json_file.exists():
    with open(json_file) as f:
        saved_results = json.load(f)
    print(f"Loaded results from {json_file}")
    print(f"  Journals analyzed: {len(saved_results['journals'])}")
else:
    print(f"No results file found at {json_file}")
    print("Run the analysis first.")

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


sizes = sorted([c["size"] for c in saved_results["communities"]], reverse=True)
n_comms     = len(sizes)
total_nodes = sum(sizes)
giant       = sizes[0]
median_sz   = float(np.median(sizes))
giant_ratio = giant / median_sz

print(f"Communities   : {n_comms}")
print(f"Giant         : {giant:,}  ({100*giant/total_nodes:.1f}%)")
print(f"#2            : {sizes[1]:,}")
print(f"Median        : {median_sz:.0f}")
print(f"Giant/median  : {giant_ratio:.1f}×  (healthy ≈ 3–5×)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(range(1, min(51, n_comms+1)), sizes[:50], color="#1a6faf", edgecolor="white", lw=0.4)
ax1.axhline(median_sz, color="tomato", lw=1.5, linestyle="--", label=f"Median ({median_sz:.0f})")
ax1.set_yscale("log")
ax1.set_xlabel("Community rank"); ax1.set_ylabel("Size (log)")
ax1.set_title("Top 50 community sizes"); ax1.legend()

non_giant = sorted(sizes[1:])
ax2.plot(non_giant, np.arange(1, len(non_giant)+1)/len(non_giant), color="#1a6faf", lw=1.8)
ax2.axvline(200, color="tomato", lw=1.2, linestyle="--", label="min_size=200")
ax2.axvline(50,  color="orange", lw=1.2, linestyle=":",  label="min_size=50")
ax2.set_xlabel("Community size"); ax2.set_ylabel("Cumulative fraction")
ax2.set_title("CDF (giant excluded)"); ax2.legend()

plt.tight_layout()
plt.savefig("scope_drift_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# # ============================================================
# # PARAMETER TUNING - Test different thresholds automatically
# # ============================================================
# # Run this cell to test multiple parameter combinations and find the best clustering

# import subprocess
# import sys

# # Quick mode tests 8 combinations, full mode tests 144
# mode = "--quick"  # Change to "--full" for exhaustive search

# print("Running parameter tuning...")
# print(
#     "This will test multiple configurations and report which gives the best clustering.\n"
# )

# result = subprocess.run(
#     [sys.executable, "src/tune_clustering.py", mode],
#     cwd=str(Path().absolute()),
#     capture_output=False,
# )

# print("\nDone! Check output/tuning_results.json for full results.")

In [ ]:
print('test')